In [1]:
%cd ..
import pandas as pd
import re
import src.bench as bench

from tqdm.notebook import tqdm
from src.utils.model_generate import opeanai_model_generate_single_answer


/home/kalisto/Documents/9_sem/BigDataAndML/Lossy-RAG


In [2]:
questions = pd.read_csv("data/questions.csv")
questions_paraphrased = pd.read_csv("data/questions_paraphrased.csv")
answers = pd.read_csv("data/answers.csv")

In [3]:
questions = questions[questions["role"] == "valid"]
questions_paraphrased = questions_paraphrased[questions_paraphrased["role"] == "valid"]

questions.reset_index()
questions_paraphrased.reset_index()
answers.reset_index()

answers = answers.groupby("orig_qid", as_index=False).agg(list)

pass


In [4]:
questions = questions[questions["orig_qid"].isin(questions_paraphrased["orig_qid"])]
questions = questions[~questions["is_paraphrase"]]

questions = pd.merge(questions, answers, on="orig_qid", how="left")
questions_paraphrased = pd.merge(questions_paraphrased, answers, on="orig_qid", how="left")

questions = questions.rename(columns={"text": "answer_text", "start": "answer_start_pos"})
questions_paraphrased = questions_paraphrased.rename(columns={"text": "answer_text", "start": "answer_start_pos"})

In [5]:
questions = questions.sort_values("orig_qid").reset_index(drop=True)
questions_paraphrased = questions_paraphrased.sort_values("orig_qid").reset_index(drop=True)

In [6]:
questions_paraphrased

,qid,squad_id,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos
0,100417,NaN,88027,valid,True,18914,Where did the Panthers train ahead of the Supe...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]"
1,100463,NaN,88051,valid,True,18916,What hue were the balls given to many secondar...,"[gold, Gold]","[8, 357]"
2,100173,NaN,88112,valid,True,18920,Which program connects area firms with deals t...,[Business Connect],[164]
3,100518,NaN,88115,valid,True,18921,What is the Host Committee's philanthropic pro...,"[the 50 fund, 50 fund]","[200, 204]"
4,100467,NaN,88144,valid,True,18923,With how many TV networks does the NFL have ai...,[three],[84]
...,...,...,...,...,...,...,...,...,...
495,100330,NaN,98112,valid,True,20945,What is required to squeeze negatively charged...,"[energy, energy]","[579, 539]"
496,100287,NaN,98116,valid,True,20946,What interaction acts between protons and neut...,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]"
497,100582,NaN,98131,valid,True,20949,What ultimately results from attaching ever mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]"
498,100398,NaN,98133,valid,True,20950,To what objects did the physics of Newton apply?,"[idealized point particles, idealized point pa...","[100, 100]"


In [7]:
questions

,qid,squad_id,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos
0,88027,56d9a6b4dc89441400fdb6a0,88027,valid,False,18914,Where was the practice place the Panthers used...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]"
1,88051,56bf52483aeaaa14008c95c8,88051,valid,False,18916,What color were the footballs handed out to a ...,"[gold, Gold]","[8, 357]"
2,88112,56d9b5ffdc89441400fdb720,88112,valid,False,18920,What is the name of the service that gets loca...,[Business Connect],[164]
3,88115,56bebec43aeaaa14008c934a,88115,valid,False,18921,What is the name of the Host Committee's chari...,"[the 50 fund, 50 fund]","[200, 204]"
4,88144,56d71ed50d65d2140019837d,88144,valid,False,18923,How many broadcast television partners does th...,[three],[84]
...,...,...,...,...,...,...,...,...,...
495,98112,5737958b1c456719005744c5,98112,valid,False,20945,What is needed to pack electrons densely toget...,"[energy, energy]","[579, 539]"
496,98116,573796edc3c5551400e51f35,98116,valid,False,20946,What is the force between nucleons?,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]"
497,98131,57379ed81c456719005744d8,98131,valid,False,20949,What is the final effect of adding more and mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]"
498,98133,5737a0acc3c5551400e51f47,98133,valid,False,20950,What did Newton's mechanics affect?,"[idealized point particles, idealized point pa...","[100, 100]"


In [9]:
from functools import partial

model_name = "Swift-1.5-Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS"
question_num = 10

gen = partial(
    opeanai_model_generate_single_answer,
    model_name=model_name,
    max_new_tokens=4096,
    reasoning_budget_tokens=3072,
    do_sample=True,
    temperature=1,
    top_p=0.95,
)

score, elapsed = bench.run_squad(gen, questions.head(question_num), responses_path=f"log/model_answers/squad_{model_name}.jsonl")
score_p, elapsed_p = bench.run_squad(gen, questions_paraphrased.head(question_num), responses_path=f"log/model_answers/squad_p_{model_name}.jsonl")

print("Num questions",  question_num, "\n", "Score:",score, "\n", "Score paraphrase", score_p)

squad_Swift-1.5-Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS:   0%|          | 0/10 [00:00<?, ?req/s]

squad_p_Swift-1.5-Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS:   0%|          | 0/10 [00:00<?, ?req/s]

Num questions 10 
 Score: 30.0 
 Score paraphrase 10.0
